In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.backends.cudnn as cudnn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torchvision.models as models
from sklearn.utils.class_weight import compute_class_weight
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler
import numpy as np
import os

# ---------------- Setup ----------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type == 'cuda':
    cudnn.benchmark = True

# ---------------- Hyperparameters ----------------
BATCH_SIZE   = 32
EPOCHS       = 30
INITIAL_LR   = 0.001
WEIGHT_DECAY = 1e-4
NUM_CLASSES  = 7
RANDOM_SEED  = 42

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ---------------- Datasets & Loaders ----------------
train_val_dataset = datasets.ImageFolder(root="./train", transform=train_transforms)
train_size = int(0.75 * len(train_val_dataset))
val_size   = len(train_val_dataset) - train_size
train_dataset, val_dataset = random_split(
    train_val_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

test_dataset = datasets.ImageFolder(root="./test", transform=val_test_transforms)

# Use multiple workers and pin memory on CUDA
num_workers = min(4, os.cpu_count() if os.cpu_count() is not None else 1)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=num_workers, pin_memory=(DEVICE.type=='cuda'))
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=(DEVICE.type=='cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=num_workers, pin_memory=(DEVICE.type=='cuda'))

# ---------------- Class Weights ----------------
all_labels = [s[1] for s in train_val_dataset.samples]
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(all_labels),
    y=all_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import GradScaler, autocast
from torchvision import models
from tqdm import tqdm

def build_model(num_classes, device):
    model = models.resnet18(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False
    num_ftrs = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(num_ftrs, 256),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(256, num_classes)
    )
    return model.to(device)

def train_one_epoch(model, dataloader, criterion, optimizer, scaler, device, epoch, total_epochs):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    loader = tqdm(dataloader, desc=f"Epoch {epoch}/{total_epochs}", unit='batch')
    
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        with autocast():
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)
        loader.set_postfix(loss=running_loss/total, acc=correct/total)

    return running_loss / total, correct / total

def validate(model, dataloader, criterion, device):
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, lbls in dataloader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            with autocast():
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
            val_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == lbls).sum().item()
            val_total += lbls.size(0)

    return val_loss / val_total, val_correct / val_total

def train_model(train_loader, val_loader, num_classes, class_weights, device,
                initial_lr=1e-3, weight_decay=1e-4, epochs=10, save_path="cnn_best.pth"):
    
    model = build_model(num_classes, device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.fc.parameters(), lr=initial_lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = GradScaler()
    best_val_loss = float('inf')

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, device, epoch, epochs
        )
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch}/{epochs} | LR: {current_lr:.6f} | "
              f"Train: loss={train_loss:.4f}, acc={train_acc:.4f} | "
              f"Val: loss={val_loss:.4f}, acc={val_acc:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print("✅ Saved best model")

    return model


In [ ]:
model = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    num_classes=NUM_CLASSES,
    class_weights=class_weights,
    device=DEVICE,
    initial_lr=INITIAL_LR,
    weight_decay=WEIGHT_DECAY,
    epochs=EPOCHS,
    save_path="cnn_nour.pth"
)


In [ ]:
model.load_state_dict(torch.load("best_resnet.pth"))
model.eval()
correct = total = 0
with torch.no_grad():
    for imgs, lbls in test_loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        with autocast():
            outputs = model(imgs)
        preds = outputs.argmax(dim=1)
        correct += (preds == lbls).sum().item()
        total += lbls.size(0)

test_acc = correct / total
print(f"\n🎯 Final Test Accuracy: {test_acc * 100:.2f}%")